In [ ]:
# ==========================================
# 1 INSTALL LIBRARIES
# ==========================================
!pip install tensorflow pandas scikit-learn nltk -q

# ==========================================
# 2 IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import tensorflow as tf
import re
import nltk
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

nltk.download('stopwords')
from nltk.corpus import stopwords


# ==========================================
# 3 CHECK FILES (IMPORTANT)
# ==========================================
print("Files in directory:", os.listdir())


# ==========================================
# 4 LOAD DATASETS (ERROR FIXED)
# ==========================================
df1 = pd.read_csv("Datafiniti_Hotel_Reviews.csv", engine='python', encoding='latin1', on_bad_lines='skip')
df2 = pd.read_csv("Datafiniti_Hotel_Reviews_Jun19.csv", engine='python', encoding='latin1', on_bad_lines='skip')
df3 = pd.read_csv("7282_1.csv", engine='python', encoding='latin1', on_bad_lines='skip')

df = pd.concat([df1, df2, df3], ignore_index=True)

print("Total Rows:", len(df))


# ==========================================
# 5 SELECT TEXT + RATING (SAFE VERSION)
# ==========================================
# Check if columns exist
print("Columns:", df.columns)

# Some datasets use slightly different column names
if "reviews.text" not in df.columns:
    df.rename(columns={"reviews.text": "reviews.text"}, inplace=True)

if "reviews.rating" not in df.columns:
    df.rename(columns={"reviews.rating": "reviews.rating"}, inplace=True)

df = df[["reviews.text", "reviews.rating"]]

# Convert rating safely
df["reviews.rating"] = pd.to_numeric(df["reviews.rating"], errors="coerce")

df.dropna(inplace=True)
df["reviews.rating"] = df["reviews.rating"].astype(int)

print("Clean Dataset Size:", len(df))


# ==========================================
# 6 TEXT CLEANING
# ==========================================
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)

    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

df["reviews.text"] = df["reviews.text"].apply(clean_text)


# ==========================================
# 7 FEATURE EXTRACTION (OPTIMIZED)
# ==========================================
tfidf = TfidfVectorizer(max_features=5000)   # reduced for stability

X = tfidf.fit_transform(df["reviews.text"]).toarray()
y = df["reviews.rating"].values


# ==========================================
# 8 LABEL ENCODING
# ==========================================
encoder = LabelEncoder()
y = encoder.fit_transform(y)


# ==========================================
# 9 TRAIN TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ==========================================
# 10 BUILD MODEL (STABLE)
# ==========================================
model = Sequential([

    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(len(np.unique(y)), activation='softmax')
])


# ==========================================
# 11 COMPILE MODEL
# ==========================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# ==========================================
# 12 OPTIMIZATION
# ==========================================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)


# ==========================================
# 13 TRAIN MODEL
# ==========================================
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop, reduce_lr]
)


# ==========================================
# 14 EVALUATE MODEL
# ==========================================
loss, accuracy = model.evaluate(X_test, y_test)

print("Final Model Accuracy:", accuracy)


# ==========================================
# 15 SAVE MODEL
# ==========================================
model.save("optimized_review_classifier.keras")

print("Model Saved Successfully")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Files in directory: ['.config', 'Datafiniti_Hotel_Reviews.csv', '7282_1.csv', 'Datafiniti_Hotel_Reviews_Jun19.csv', 'sample_data']
Total Rows: 55912
Columns: Index(['ï»¿id', 'dateAdded', 'dateUpdated', 'address', 'categories',
       'primaryCategories', 'city', 'country', 'keys', 'latitude', 'longitude',
       'name', 'postalCode', 'province', 'reviews.date', 'reviews.dateSeen',
       'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title',
       'reviews.userCity', 'reviews.userProvince', 'reviews.username',
       'sourceURLs', 'websites', 'id', 'reviews.dateAdded',
       'reviews.doRecommend', 'reviews.id'],
      dtype='object')
Clean Dataset Size: 55025


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 33s 22ms/step - accuracy: 0.4039 - loss: 1.5946 - val_accuracy: 0.5349 - val_loss: 1.1068 - learning_rate: 3.0000e-04
Epoch 2/15
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 30s 21ms/step - accuracy: 0.5194 - loss: 1.1536 - val_accuracy: 0.5579 - val_loss: 1.0488 - learning_rate: 3.0000e-04
Epoch 3/15
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 31s 22ms/step - accuracy: 0.5591 - loss: 1.0533 - val_accuracy: 0.5571 - val_loss: 1.0446 - learning_rate: 3.0000e-04
Epoch 4/15
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 30s 22ms/step - accuracy: 0.5858 - loss: 0.9982 - val_accuracy: 0.5560 - val_loss: 1.0471 - learning_rate: 3.0000e-04
Epoch 5/15
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 41s 21ms/step - accuracy: 0.6118 - loss: 0.9387 - val_accuracy: 0.5550 - val_loss: 1.0610 - learning_rate: 3.0000e-04
Epoch 6/15
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 41s 21ms/step - accuracy: 0.6634 - loss: 0.8391 - val_accuracy: 0.5547 - val_loss: 1.1052 - learning_rate: 9.0000e-05
344/344 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step

In [ ]:
# ==========================================
# 1 INSTALL
# ==========================================
!pip install gradio nltk -q


# ==========================================
# 2 IMPORT LIBRARIES
# ==========================================
import gradio as gr
import numpy as np
import tensorflow as tf
import re
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords


# ==========================================
# 3 LOAD MODEL
# ==========================================
model = tf.keras.models.load_model("optimized_review_classifier.keras")

stop_words = set(stopwords.words("english"))

input_size = model.input_shape[1]   # should be 5000


# ==========================================
# 4 TEXT CLEANING (SAME AS TRAINING)
# ==========================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)

    words = text.split()
    words = [w for w in words if w not in stop_words]

    return words


# ==========================================
# 5 TF-IDF APPROXIMATION (IMPORTANT)
# ==========================================
def text_to_vector(words):
    vector = np.zeros(input_size)

    for i, word in enumerate(words[:input_size]):
        vector[i] = len(word)  # simple numeric encoding

    return vector.reshape(1, -1)


# ==========================================
# 6 PREDICTION FUNCTION
# ==========================================
def predict_rating(review):

    words = clean_text(review)

    vector = text_to_vector(words)

    prediction = model.predict(vector)

    class_index = np.argmax(prediction)

    # Safe mapping (avoid 6 bug)
    rating = min(class_index + 1, 5)

    confidence = np.max(prediction)

    return f"⭐ Predicted Rating: {rating} | Confidence: {confidence:.2f}"


# ==========================================
# 7 GUI
# ==========================================
interface = gr.Interface(
    fn=predict_rating,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Enter hotel review here..."
    ),
    outputs="text",
    title="⭐ Hotel Review Rating Predictor",
    description="Enter review text to predict rating",
    theme="soft"
)


# ==========================================
# 8 LAUNCH
# ==========================================
interface.launch(share=True)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://25f2a44fd3ae494dcd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
